In [1]:
import os
import sys
from pathlib import Path

# Megkeressük a projekt gyökerét (egy szinttel feljebb a notebook mappájától)
project_root = Path(os.getcwd()).resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import config  # Most már a gyökérből importál
import wandb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Login és Init a config használatával
wandb.login()

run = wandb.init(
    project=config.WANDB_PROJECT,
    name=config.WANDB_NAME,
    job_type="eda"
)

print(f"WandB initialized: {config.WANDB_PROJECT} / {config.WANDB_NAME}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Currently logged in as: pistuka700 (pistuka700-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /work/scripts/wandb/run-20260404_183933-k6b3b1fp
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run eda-exploration


wandb: ⭐️ View project at https://wandb.ai/pistuka700-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/tooth-detection-project


wandb: 🚀 View run at https://wandb.ai/pistuka700-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/tooth-detection-project/runs/k6b3b1fp


WandB initialized: tooth-detection-project / eda-exploration


In [2]:
from tqdm.notebook import tqdm

from classification_pipeline import (
    ClassificationDownloadConfig,
    ToothCropDataset,
    build_classification_image_pipeline,
    build_classification_records_from_masks,
    build_classification_resize_pipeline,
    load_or_download_classification_dataset,
    split_grouped_records as split_classification_records,
    summarize_binary_labels,
)
from detection_pipeline import (
    AugmentedToothDetectionDataset,
    DetectionDownloadConfig,
    ToothDetectionDataset,
    build_detection_records,
    build_detection_train_pipeline,
    load_or_download_detection_dataset,
    split_grouped_records as split_detection_records,
)

from utils.data_split import split_records_by_subset
from utils.display_item import to_display_image

api_key = os.getenv("ROBOFLOW_API_KEY")

# Load Detection Dataset
_, det_coco, det_dirs = load_or_download_detection_dataset(DetectionDownloadConfig(api_key=api_key))
det_records, _ = build_detection_records(det_coco, det_dirs)
det_train, det_val, det_test = split_records_by_subset(det_records)

# Load Classification Dataset
_, cls_coco, cls_dirs = load_or_download_classification_dataset(ClassificationDownloadConfig(api_key=api_key))
cls_records = build_classification_records_from_masks(cls_coco, cls_dirs)
cls_train, cls_val, cls_test = split_records_by_subset(cls_records)

# Create Dataset objects for visualization
det_ds = ToothDetectionDataset(det_train, image_size=640)
det_aug_ds = AugmentedToothDetectionDataset(det_ds, build_detection_train_pipeline())

resize_pipe = build_classification_resize_pipeline(224)
classification_ds = ToothCropDataset(cls_train, image_size=224, resize_transform=resize_pipe)
classification_aug_ds = ToothCropDataset(cls_train, image_size=224, resize_transform=resize_pipe, 
                                        image_transform=build_classification_image_pipeline())

print(f"Loaded: {len(det_train)} detection and {len(cls_train)} classification images.")

Loaded: 4361 detection and 4171 classification images.


In [3]:
# Detailed summary table for detection samples
detection_table = wandb.Table(columns=["file_name", "tooth_count", "avg_relative_area", "out_of_bounds", "contains_massive_bbox", "status"])

for i in tqdm(range(len(det_ds)), desc="Analyzing Detection Dataset"):
    img_tensor, target = det_ds[i]
    boxes = target["boxes"].numpy()
    file_name = target["file_name"]
    
    tooth_count = len(boxes)
    areas = []
    
    out_of_bounds_found = False
    massive_bbox_found = False
    
    for box in boxes:
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        rel_area = (w * h) / (640 * 640)
        areas.append(rel_area)
        
        # Check 1: Is the bbox outside the 640x640 canvas?
        if x1 < 0 or y1 < 0 or x2 > 640 or y2 > 640:
            out_of_bounds_found = True
            
        # Check 2: Is a single tooth covering more than 15% of the total image? 
        # (Unrealistic for a single tooth in a full panoramic X-ray)
        if rel_area > 0.15:
            massive_bbox_found = True
        
        # Log individual bbox stats for histograms
        wandb.log({
            "detection/bbox_aspect_ratio": w / (h + 1e-6),
            "detection/bbox_area_relative": rel_area
        })

    # Logic for the final status flag
    if out_of_bounds_found:
        status = "ERROR: Out of Bounds"
    elif massive_bbox_found:
        status = "WARNING: Giant BBox detected"
    elif tooth_count == 0:
        status = "ERROR: No annotations"
    elif tooth_count > 32:
        status = "WARNING: Too many teeth"
    else:
        status = "OK"

    detection_table.add_data(
        file_name, 
        tooth_count, 
        np.mean(areas) if areas else 0, 
        out_of_bounds_found, 
        massive_bbox_found, 
        status
    )
    
    wandb.log({"detection/teeth_per_image": tooth_count})

wandb.log({"detection/summary_table": detection_table})

print(f"Detection analysis complete.")

Analyzing Detection Dataset:   0%|          | 0/4361 [00:00<?, ?it/s]

Detection analysis complete.


In [4]:
import re
import os

def get_base_id(filename):
    """
    Strips Roboflow hashes and extensions to get a clean ID.
    Example: '3683190000-jpg_png_jpg.rf.d94e189822f1af7cd5e9e59839e7f307.jpg' 
             -> '3683190000-jpg_png_jpg'
    """
    base = os.path.basename(filename)
    # Regex: everything before the first '.rf.' or the last extension
    match = re.split(r'\.rf\.', base, flags=re.IGNORECASE)
    return match[0].strip()

# --- Dataset B: Classification & Per-Image Statistics ---

# Build a CLEANED Detection Map (using only base IDs)
# We use det_records (the full list from Cell 2) to ensure coverage
detection_map = {get_base_id(r["file_name"]): len(r["boxes"]) for r in det_records}

# Aggregate Classification results
per_image_stats = {}
unmatched_files = []

for record in cls_train:
    base_id = get_base_id(record["file_name"])
    
    if base_id not in per_image_stats:
        # Lookup the total teeth count using the clean ID
        total_teeth = detection_map.get(base_id, 0)
        
        if total_teeth == 0:
            unmatched_files.append(record["file_name"])
            
        per_image_stats[base_id] = {
            "carious_count": 0,
            "total_annotated_teeth": total_teeth,
            "original_name": record["file_name"] # keeping one name for the table
        }
    
    per_image_stats[base_id]["carious_count"] += int(record["label"])

# 3. Validation Output
matched_count = sum(1 for s in per_image_stats.values() if s["total_annotated_teeth"] > 0)
print(f"✅ Successfully matched: {matched_count} images")
print(f"❌ Still unmatched: {len(per_image_stats) - matched_count} images")

if unmatched_files:
    print(f"Sample unmatched file: {unmatched_files[0]}")
    print(f"Attempted ID: {get_base_id(unmatched_files[0])}")

# 4.Log to WandB Table
per_image_table = wandb.Table(columns=[
    "source_id", 
    "carious_teeth_count", 
    "total_teeth_count", 
    "caries_percentage"
])

for base_id, stats in per_image_stats.items():
    c_count = stats["carious_count"]
    t_count = stats["total_annotated_teeth"]
    ratio = c_count / t_count if t_count > 0 else 0
    
    per_image_table.add_data(base_id, c_count, t_count, ratio)
    
    # Log individual values for Histograms
    wandb.log({
        "classification/per_image_carious_count": c_count,
        "classification/per_image_total_teeth": t_count
    })

# Global Summary Logs
cls_summary = summarize_binary_labels(cls_train)
wandb.log({
    "classification/total_caries_ratio": cls_summary["positive"] / cls_summary["total"],
    "classification/per_image_summary_table": per_image_table
})

# Overall Class Distribution Bar Chart
dist_data = [["Carious (1)", cls_summary["positive"]], ["Healthy (0)", cls_summary["negative"]]]
dist_table = wandb.Table(data=dist_data, columns=["Class", "Count"])
wandb.log({"classification/class_distribution_plot": wandb.plot.bar(dist_table, "Class", "Count", title="Overall Class Distribution")})

print("Analysis complete. Check WandB for the updated charts!")

✅ Successfully matched: 3412 images
❌ Still unmatched: 0 images


Analysis complete. Check WandB for the updated charts!


In [5]:
# Table to compare original images with their augmented versions
viz_table = wandb.Table(columns=["ID", "Original_Image", "Augmented_Image", "Task"])

# 5 samples from Detection
for i in range(5):
    img_orig, _ = det_ds[i]
    img_aug, _ = det_aug_ds[i]
    viz_table.add_data(f"det_sample_{i}", 
                       wandb.Image(to_display_image(img_orig)), 
                       wandb.Image(to_display_image(img_aug)), 
                       "Detection")

# 5 samples from Classification
for i in range(5):
    img_orig, label, meta = classification_ds[i]
    img_aug, _, _ = classification_aug_ds[i]
    viz_table.add_data(meta['file_name'], 
                       wandb.Image(to_display_image(img_orig), caption=f"Label: {label}"), 
                       wandb.Image(to_display_image(img_aug)), 
                       "Classification")

wandb.log({"verification/augmentation_samples": viz_table})

print(f"Image verification section complete.")

Image verification section complete.


In [6]:
# Simulated training loop to test dashboard appearance
for epoch in range(1, 21):
    # Simulated loss reduction and accuracy improvement
    train_loss = 0.6 * (0.85 ** epoch) + np.random.normal(0, 0.01)
    val_accuracy = 0.65 + 0.3 * (1 - 0.9 ** epoch)
    
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "val/accuracy": val_accuracy,
        "val/loss": train_loss * 1.05  # Simulated validation loss
    })

# Simulated misclassification visualization for debugging
debug_table = wandb.Table(columns=["epoch", "image", "predicted", "actual"])
sample_img, label, _ = classification_ds[0]
debug_table.add_data(20, wandb.Image(to_display_image(sample_img)), 1, 0) # False Positive simulation

wandb.log({"debug/misclassifications": debug_table})

# Finish WandB run
run.finish()
print("Run finished. Check results on wandb.ai")

wandb: uploading artifact run-k6b3b1fp-classificationper_image_summary_table-rrLkuQ; updating run metadata; uploading artifact run-k6b3b1fp-classificationclass_distribution_plot_table-KkNHoQ; uploading artifact run-k6b3b1fp-verificationaugmentation_samples-GWMF4w; uploading artifact run-k6b3b1fp-debugmisclassifications-2482LQ


wandb: uploading artifact run-k6b3b1fp-classificationper_image_summary_table-rrLkuQ; uploading artifact run-k6b3b1fp-classificationclass_distribution_plot_table-KkNHoQ; uploading artifact run-k6b3b1fp-verificationaugmentation_samples-GWMF4w; uploading artifact run-k6b3b1fp-debugmisclassifications-2482LQ


wandb: uploading artifact run-k6b3b1fp-verificationaugmentation_samples-GWMF4w; uploading artifact run-k6b3b1fp-debugmisclassifications-2482LQ


wandb: uploading artifact run-k6b3b1fp-verificationaugmentation_samples-GWMF4w; uploading artifact run-k6b3b1fp-debugmisclassifications-2482LQ; uploading wandb-summary.json; uploading config.yaml; uploading media/table/detection/summary_table_53508_bff362595235bcc99209.table.json (+ 5 more)


wandb: uploading artifact run-k6b3b1fp-verificationaugmentation_samples-GWMF4w; uploading media/table/detection/summary_table_53508_bff362595235bcc99209.table.json; uploading media/table/classification/per_image_summary_table_56921_1c1eb59b23b2ca9a45d1.table.json


...

wandb: uploading history steps 50518-56944, summary, console lines 2-6


wandb: 
wandb: Run history:
wandb: classification/per_image_carious_count ▁▁▁▁▁▁▁▁▁▅▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   classification/per_image_total_teeth ▂█▂▂█▁█▂▂▇▂▁▂▂▂█▂▂▂▂▂▁▂▂▂▁▂▂▂▁▁▂▂▁▁▂▂▂▂▂
wandb:      classification/total_caries_ratio ▁
wandb:           detection/bbox_area_relative ▃▂▄▁▃▆▅█▂▄▆▂▄█▃▂▄▃▄▅▃▇▂▇▃▄▇▆▃▅▁▄▄▃▃▂▅▂▄▄
wandb:            detection/bbox_aspect_ratio ▃▇▂▂▂▆▃▂▃▅▂▄▁▃▅▃▁▃▂▃▄▂▃▂▇▃▁▃█▁▃▅▇▅▃▁▃▄▁█
wandb:              detection/teeth_per_image ▂▂▇▂▂▆▁▁▂▂▂▂▂▁▁▂▂▂▂▁▁▂▂▂▂▁▇▁▂▂▂▂█▂▂█▂▇█▂
wandb:                                  epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:                             train/loss █▇▆▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁
wandb:                           val/accuracy ▁▂▃▃▄▄▅▅▆▆▆▇▇▇▇▇████
wandb:                               val/loss █▇▆▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb: classification/per_image_carious_count 1
wandb:   classification/per_image_total_teeth 8
wandb:      classification/total_caries_ratio 1
wandb:           detection/bbox_area_relative 0.0

wandb: 🚀 View run eda-exploration at: https://wandb.ai/pistuka700-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/tooth-detection-project/runs/k6b3b1fp
wandb: ⭐️ View project at: https://wandb.ai/pistuka700-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/tooth-detection-project
wandb: Synced 5 W&B file(s), 5 media file(s), 30 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260404_183933-k6b3b1fp/logs


Run finished. Check results on wandb.ai
